# 04 — Stack Ensemble

Train a meta-model on the OOF predictions from each model class.
Log the candidate run pinned to the exact base-model versions from
03's `base_pins.json`, then write a manifest (candidate run id +
model URI) so 05 loads exactly this run — never a latest lookup.

In [ ]:
from src.utils import load_env

load_env()

from src.constants import CANDIDATE_MANIFEST, DATA_PROCESSED

input_dir = str(DATA_PROCESSED)

random_state = 42

model_names = ["linear", "gbdt", "nn"]

candidate_manifest = str(CANDIDATE_MANIFEST)

In [ ]:
import json
import numpy as np
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

In [ ]:
# ── Load OOF predictions ──
oof_preds = pd.read_parquet(f"{input_dir}/oof_preds.parquet").values
y_train = pd.read_parquet(f"{input_dir}/y_train_full.parquet").values.ravel()
print(f"OOF predictions shape: {oof_preds.shape}")

In [ ]:
# ── Train meta-model ──
meta = LogisticRegression(random_state=random_state)
meta.fit(oof_preds, y_train)

print("Meta-model coefficients:")
for name, coef in zip(model_names, meta.coef_[0], strict=False):
    print(f"  {name:8s} {coef:.4f}")

In [ ]:
# ── Log pinned candidate to MLflow (no registration/promotion) ──
with open(f"{input_dir}/base_pins.json") as f:
    base_pins = json.load(f)
with open(f"{input_dir}/aux_pins.json") as f:
    aux_pins = json.load(f)
with open(f"{input_dir}/feature_pins.json") as f:
    feature_pins = json.load(f)

mlflow.set_experiment("stacked_ensemble")
with mlflow.start_run():
    # Exact lineage of every base model this candidate was trained from, so
    # 05 tags the promoted version and deploy resolves exactly these versions.
    # No aliases: registered name + version + run ID + model URI are the contract.
    for name in model_names:
        pin = base_pins[name]
        mlflow.log_param(f"base_{name}_registered_name", pin["registered_model_name"])
        mlflow.log_param(f"base_{name}_version", pin["version"])
        mlflow.log_param(f"base_{name}_run_id", pin["run_id"])
        mlflow.log_param(f"base_{name}_model_uri", pin["model_uri"])
    mlflow.log_metrics(
        {f"weight_{name}": coef for name, coef in zip(model_names, meta.coef_[0], strict=False)}
    )
    mlflow.sklearn.log_model(meta, "stacked_ensemble")
    run = mlflow.active_run()
    assert run is not None
    run_id = run.info.run_id
    print(f"Candidate logged to MLflow run: {run_id}")

In [ ]:
# ── Manifest for 05: exact candidate run + full lineage, no latest lookup ──
manifest = {
    "candidate_run_id": run_id,
    "model_uri": f"runs:/{run_id}/stacked_ensemble",
    "artifact_path": "stacked_ensemble",
    "base_pins": base_pins,
    "aux_pins": aux_pins,
    "feature_pins": feature_pins,
}
with open(candidate_manifest, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest written to {candidate_manifest}")
print(f"  candidate_run_id: {run_id}")
print(f"  model_uri:        runs:/{run_id}/stacked_ensemble")
print(
    f"  lineage: {len(base_pins)} base classes, {len(aux_pins)} aux keys, "
    f"{len(feature_pins)} feature keys"
)

In [ ]:
print("Done. Run 05_evaluate.ipynb for full evaluation on the test set.")